# SentiTriage-ES — Fine-tuning de BETO y XLM-R (Google Colab)

Este notebook ejecuta en GPU los experimentos 2 y 3 del artículo con **el mismo código y el mismo protocolo** que el baseline del repositorio (`src/sentitriage/finetune.py` y `evaluate.py`).

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.

Tiempo aproximado con T4: BETO ≈ 6–10 min, XLM-R ≈ 8–12 min.

Al final se descarga `sentitriage_colab_outputs.zip` con:
- `results/beto-sentitriage/` y `results/xlmr-sentitriage/` (métricas, predicciones, figuras)
- `models/beto-sentitriage/` (modelo exportado que carga el servicio)

Descomprímelo en la raíz del repositorio local.

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Código del repositorio

- Si el repo ya está en GitHub, clónalo (cambia la URL).
- Si no, sube `sentitriage-es.zip` (el repositorio comprimido) cuando la celda lo pida.

In [ ]:
import os, zipfile
REPO_URL = ""  # p. ej. "https://github.com/Rosa-SDev/sentitriage-es.git" cuando esté publicado
os.chdir("/content")
if not os.path.isdir("/content/sentitriage-es"):
    if REPO_URL:
        !git clone -q $REPO_URL sentitriage-es
    else:
        from google.colab import files
        up = files.upload()                      # selecciona sentitriage-es.zip
        name = next(iter(up))
        zipfile.ZipFile(name).extractall("/content")
os.chdir("/content/sentitriage-es")
!git log --oneline -5 2>/dev/null || ls

In [ ]:
!pip install -q -r requirements.txt
import sys; sys.path.insert(0, "src")
from sentitriage.data import download, load_all
download()
{k: len(v) for k, v in load_all().items()}

## 3. Experimento 2 — BETO (modelo principal)

`dccuchile/bert-base-spanish-wwm-cased` + capa lineal sobre [CLS]. 4 épocas, lr 2e-5, batch 16, longitud máx. 128, semilla 42. Se guarda la época con mejor macro-F1 en validación.

In [ ]:
from sentitriage.finetune import run, MODELS
beto = run(MODELS["beto"], "beto", out_root=".")
{k: beto[k] for k in ["macro_f1", "macro_f1_ci95", "recall_negative", "temperature",
                      "ece_test_before", "ece_test_after", "aurc_test", "selective_test",
                      "missed_negatives_without_abstention", "missed_negatives_with_policy"]}

## 4. Experimento 3 — XLM-R base (control multilingüe, comparable con [6])

In [ ]:
xlmr = run(MODELS["xlmr"], "xlmr", out_root=".")
{k: xlmr[k] for k in ["macro_f1", "macro_f1_ci95", "recall_negative", "temperature",
                      "ece_test_after", "aurc_test", "selective_test"]}

## 5. Comparación de los tres modelos

In [ ]:
!python experiments/run_baseline.py > /dev/null
!python experiments/compare.py
!python experiments/error_analysis.py beto-sentitriage
from IPython.display import Image, display
display(Image("results/risk_coverage_all.png"))
display(Image("results/beto-sentitriage/reliability.png"))

## 6. Descargar resultados y el modelo BETO

El modelo XLM-R no se incluye en el zip para que pese menos (solo sus resultados).

In [ ]:
import shutil, os
shutil.rmtree("/content/export", ignore_errors=True)
os.makedirs("/content/export")
shutil.copytree("results", "/content/export/results")
shutil.copytree("models/beto-sentitriage", "/content/export/models/beto-sentitriage")
shutil.make_archive("/content/sentitriage_colab_outputs", "zip", "/content/export")
from google.colab import files
files.download("/content/sentitriage_colab_outputs.zip")

## 7. Usar BETO en el servicio local

```bash
# en la raíz del repo, tras descomprimir el zip
pip install -r requirements-transformers.txt
SENTITRIAGE_MODEL_DIR=models/beto-sentitriage uvicorn service.app:app --port 8000
# Windows PowerShell:  $env:SENTITRIAGE_MODEL_DIR="models/beto-sentitriage"; uvicorn service.app:app --port 8000
```